# Reproducing Tingley and Buzsaki (2019) from dandiset 000218

Paper: Tingley D, Buzsaki G (2019). Routing of hippocampal ripples to subcortical structures via the
lateral septum. Neuron 105, 138-149. [doi:10.1016/j.neuron.2019.10.012](https://doi.org/10.1016/j.neuron.2019.10.012).
The full text was used: the accepted author manuscript in PubMed Central (PMC6952543). Its
supplementary figures were not available, so claims on Figures S1 to S5 rest on the main text.

Dandiset: [DANDI:000218 version 0.250624.0439](https://dandiarchive.org/dandiset/000218/0.250624.0439),
98 NWB files (1.5 TB) from 6 rats, streamed from the archive. The paper cites no version (it says
the data would go to CRCNS). There are two published versions, 0.220131.1608 and 0.250624.0439;
they hold the same 98 files with the same blob ids (checked below) and differ only in contributor
metadata, so the most recent one is pinned.

The files hold the LFP at 1250 Hz on every channel, sorted units with a region label in most files,
sleep states, epochs and trials. They hold no detected events. LS high-frequency oscillations
(HFOs) and CA1 ripples are detected here from the LFP, following the Methods and the buzcode
detector they name (see `ls_hfo.py`). The paper's own code repository holds no analysis of this
paper apart from the network model of Figure 8.

| Claim | Tested here |
|---|---|
| dataset-composition: 6 rats, 1989 LS neurons, 37 CA1+LS and 25 CA3+LS recordings | tested |
| fig1c-hfo-each-animal: HFOs with increased LS firing in each of 6 animals | tested |
| fig1c-event-count: 29,955 HFOs | tested |
| figs1-participation: 16% of LS neurons per HFO, 0.23 spikes per event | tested |
| fig1d-phase-locking: 92% of LS neurons phase-locked, trough preference | tested |
| fig1f-shorter-than-ripples: LS HFOs 27 ms against CA1 ripples 31 ms | tested |
| fig1g-slower-than-ripples: 147 against 159 Hz | tested |
| fig1h-no-amplitude-frequency-correlation | tested |
| fig2-confined-to-ls: HFOs only at LS sites | tested |
| fig2ef-intermediate-ls: power and locking peak in intermediate LS | blocked (metadata insufficient) |
| fig3-travelling-waves: propagation speeds | blocked (metadata insufficient) |
| fig4b-pre-post-unchanged: excess correlation PRE = POST | tested |
| fig4b-hfo-above-behav: HFO excess correlation above BEHAV and jitter | tested |
| fig4-sessions-few-change: 7 of 53 sessions differ | blocked after selection (too few recordings) |
| fig4c-temporal-bias: PRE and POST biases similar | blocked after selection (too few recordings) |
| fig4d-compression: above chance up to 20-fold | tested |
| figs2-post-rate-increase: rates rise in post-behavior NREM | tested |
| fig5b-nrem-and-rates: HFOs mostly in NREM, rarer than ripples | tested |
| fig5d-lag: CA1 leads LS by 6.4 ms | tested |
| fig5e-state-coupling: coupling 0.3 in NREM against 0.15 in WAKE | tested |
| fig5f-ripple-triggered-firing: higher in NREM | tested |
| fig5f-no-phase-coherence: CA1-LS coherence at chance | tested |
| fig6-optogenetic-hfos | blocked (data missing) |
| fig7e-ca1-spikes: 38% of LS neurons | tested |
| fig7e-state: 26% | tested |
| fig7e-duration: about 20% | tested |
| fig7e-ca1-depth: 23% | blocked (data missing) |
| fig7e-replay: 7.9, 6.6 and 7.1% | blocked (methods underspecified) |
| fig7e-seqnmf: 8.1% | blocked (methods underspecified) |
| fig7e-content-independent | blocked (depends on blocked claims) |
| fig8-model | blocked (other: a simulation, no recorded data) |

The criteria are in `claims/000218/10.1016-j.neuron.2019.10.012.yaml`; the outcomes and the
reasons are in `attempts/000218-tingley-2019-01.yaml`.

In [ ]:
import collections
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ls_hfo

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [ ]:
# Stream every file of the pinned version once and keep what the analyses need (about 17 min
# with 6 workers; most of it is loading each file's chunk index over the network).
started = time.time()
asset_list = sorted(ls_hfo.assets(), key=lambda a: a["path"])
results = ls_hfo.run_all([a["path"] for a in asset_list], workers=6)
print(f"dandiset {ls_hfo.DANDISET} version {ls_hfo.VERSION}: {len(asset_list)} files, "
      f"{sum(a['size'] for a in asset_list) / 1e12:.2f} TB")
print(f"streamed and processed in {(time.time() - started) / 60:.1f} min")
errors = [r for r in results if "error" in r]
print("files that failed:", len(errors))
for r in errors:
    print("  ", r["path"], r["error"])
skipped = [r for r in results if "skipped" in r]
print("files without LS electrodes (no LFP analysis):", len(skipped))
for r in skipped:
    print("  ", r["path"], dict(collections.Counter(r["meta"]["el_location"])))
R = [r for r in results if "error" not in r and "skipped" not in r]

# The two published versions hold the same files: compare paths and blob ids.
older = {a["path"]: a["blob"] for a in ls_hfo.assets("0.220131.1608")}
pinned = {a["path"]: a["blob"] for a in asset_list}
print("0.220131.1608 and the pinned version hold the same paths and blobs:", older == pinned)


## Data check

Each check of the description against the values is stated under its output.

In [ ]:
# Inventory: what each file holds.
def region_counts(values):
    return dict(sorted(collections.Counter(values).items()))

inv = []
for r in results:
    m = r["meta"]
    loc = m["units"]["location"]
    inv.append({
        "file": r["path"].split("/")[1].replace("_ecephys.nwb", "").replace("_behavior+ecephys.nwb", "")[:48],
        "subject": m["subject"], "hours": round(m["duration"] / 3600, 2),
        "electrodes": region_counts(m["el_location"]),
        "units": len(m["units"]["spikes"]),
        "unit_regions": "no location column" if loc is None else region_counts(loc),
        "states": m["states"] is not None, "epochs": m["epochs"] is not None,
        "trials": 0 if m["trials"] is None else len(m["trials"]["start"]),
    })
inv = pd.DataFrame(inv)
print(inv.to_string())
print()
print("files per subject:", inv.subject.value_counts().sort_index().to_dict())
unit_regions = collections.Counter()
for r in results:
    loc = r["meta"]["units"]["location"]
    if loc is None:
        unit_regions["(no location)"] += len(r["meta"]["units"]["spikes"])
    else:
        unit_regions.update(loc)
print("units by region label:", dict(unit_regions))
print("files whose units have no location column:",
      [r["path"].split("/")[1][:40] for r in results if r["meta"]["units"]["location"] is None and len(r["meta"]["units"]["spikes"])])
print("LFP rates (Hz):", sorted({r["meta"]["lfp_rate"] for r in results}),
      "| conversion:", sorted({r["meta"]["lfp_conversion"] for r in results}), sorted({r["meta"]["lfp_unit"] for r in results}))
print("electrode x, y, z all NaN in every file:", all(np.isnan(r["meta"]["el_xyz"]).all() for r in results))


The archive's summary gives 6 subjects, which matches the 6 rats (DT1, DT2, DT5, DT7, DT8, DT9)
and the paper's 5 rats of Tingley and Buzsaki (2018) plus one tetrode rat. DT1 holds LS channels
only, without sleep states, epochs or trials, which fits the tetrode rat used only in the
phase-locking analysis; the files do not say which rat is which.

Unit region labels: `ls`, `hpc`, `ca3`, and a few cortical, septal and fimbria labels. The `hpc`
label does not separate CA1 from other hippocampal sites. Nine files have units without a location
column; their 268 units cannot be assigned to a region and are left out of every unit analysis.
The LFP is stored at 1250 Hz as int16 with a conversion of 1.95e-7 V (0.195 uV) per step. The electrode x, y and z are NaN in every file, so the probe geometry
and its orientation in the brain are not in the files.

In [ ]:
# Duplicates, spike-time format, states, epochs and trials.
lengths = collections.Counter(r["meta"]["lfp_shape"][0] for r in results)
fps = collections.Counter(r["fingerprint"] for r in R)
print("files sharing an LFP length:", sum(v for v in lengths.values() if v > 1),
      "| files sharing the checksum of the first 10 s of their LS channel:", sum(v for v in fps.values() if v > 1))
spike_counts = collections.Counter(tuple(len(s) for s in r["meta"]["units"]["spikes"]) for r in results if len(r["meta"]["units"]["spikes"]))
print("files sharing the spike counts of all their units:", sum(v for v in spike_counts.values() if v > 1))

bad = []
for r in results:
    m = r["meta"]
    for i, s in enumerate(m["units"]["spikes"]):
        if len(s) and (np.any(np.diff(s) < 0) or s.min() < 0 or s.max() > m["duration"] + 1):
            bad.append({"file": r["path"].split("/")[1][:40], "unit": i,
                        "region": None if m["units"]["location"] is None else m["units"]["location"][i],
                        "spikes": len(s), "out_of_order": int((np.diff(s) < 0).sum()),
                        "first": round(float(s.min()), 3), "last_minus_duration": round(float(s.max() - m["duration"]), 3)})
print(f"units with spike times out of order or outside the recording: {len(bad)}")
print(pd.DataFrame(bad).to_string())

labels = collections.Counter()
cover, epoch_ok, trial_dur = [], [], []
for r in results:
    m = r["meta"]
    if m["states"] is not None:
        labels.update(m["states"]["label"])
        cover.append((m["states"]["stop"] - m["states"]["start"]).sum() / m["duration"])
    if m["epochs"] is not None and m["trials"] is not None:
        ep, tr = m["epochs"], m["trials"]
        epoch_ok.append(abs(ep["stop"][0] - tr["start"].min()) < 1 and abs(ep["stop"][1] - tr["stop"].max()) < 1)
        trial_dur.append(tr["stop"] - tr["start"])
print("sleep-state labels (intervals):", dict(labels))
print(f"fraction of each recording covered by states: {np.min(cover):.2f} to {np.max(cover):.2f} over {len(cover)} files")
print("epoch tags:", sorted({tuple(r['meta']['epochs']['tags']) for r in results if r['meta']['epochs'] is not None}))
print(f"files where the BEHAV epoch runs from the first trial start to the last trial stop: {sum(epoch_ok)} of {len(epoch_ok)}")
td = np.concatenate(trial_dur)
print(f"trial durations: median {np.median(td):.1f} s, range {td.min():.2f} to {td.max():.1f} s, {len(td)} trials")


No recording is deposited twice: no two files share an LFP length, the checksum of the first 10 s
of their LS channel, or the spike counts of all their units.

25 units have spike times out of order (1 to 2782 inversions each), a break of the NWB rule that
spike times are sorted. Spike times are sorted here before use. One `ppc` unit in DT8 ends 6.7 s
before the recording; all spikes fall inside the recordings.

Sleep states cover 82 to 100% of each recording that has them, with the labels `Non-REM`, `Awake`,
`MA` (microarousal) and `REM`. Epochs are named `before trials`, `during trials` and
`after trials`, and in every file the middle epoch runs from the first trial's start to the last
trial's stop, so the epochs are derived from the trials table. They are used as PRE, BEHAV and POST.

In [ ]:
# Channels chosen, events at each detection step, and unit firing rates.
rows = []
for r in R:
    m = r["meta"]
    row = {"file": r["path"].split("/")[1][:40], "subject": m["subject"], "ls_ch": r["channels"]["ls"],
           "hpc_ch": r["channels"]["hpc"], "emg_shanks": len(r["channels"]["emg"]),
           "bad_channels": int((~r["channel_stats"]["good"]).sum()),
           "emg>0.9 (%)": None if r["emg_summary"] is None else round(100 * r["emg_summary"]["frac_above"], 2)}
    for k, v in r["ls_events"]["counts"].items():
        row[f"ls_{k}"] = v
    if "hpc_events" in r:
        row["hpc_final"] = r["hpc_events"]["counts"]["emg"]
    rows.append(row)
det = pd.DataFrame(rows)
print(det.groupby("subject")[[c for c in det.columns if c.startswith(("ls_t", "ls_m", "ls_p", "ls_d", "ls_e", "hpc_final"))]].sum().to_string())
print(f"LS HFOs over all {len(det)} recordings with LS electrodes: {det.ls_emg.sum()}; "
      f"removed by the EMG check: {(det.ls_duration - det.ls_emg).sum()}")
print(f"recordings with a hippocampal (`hpc`) channel: {det.hpc_ch.notna().sum()}, CA1 ripples {int(det.hpc_final.sum())}")
print(f"channels dropped as bad (broadband SD outside 0.2 to 5 times the region median): {det.bad_channels.sum()} in {(det.bad_channels > 0).sum()} files")
print("EMG above 0.9, % of 0.1 s windows: median", det["emg>0.9 (%)"].median(), "max", det["emg>0.9 (%)"].max())

rates = []
for r in results:
    m = r["meta"]
    loc = m["units"]["location"]
    for i, s in enumerate(m["units"]["spikes"]):
        rates.append({"region": "(no location)" if loc is None else loc[i], "rate": len(s) / m["duration"]})
rates = pd.DataFrame(rates)
print("unit firing rates (Hz) by region:")
print(rates.groupby("region").rate.describe(percentiles=[0.5, 0.9]).round(2).to_string())
print("units above 50 Hz:", rates.groupby("region").rate.apply(lambda x: int((x > 50).sum())).to_dict())


Detection counts are summed by animal for each step of the detector: threshold crossings, after
merging, after the 5 SD peak test, after the duration limits, and after the EMG check. The EMG check
removes 19,684 LS events (23% of those that pass the other steps). LS HFO counts differ widely
between animals; DT7 contributes more than half of them (38,008 of 67,186). Only 5 channels in 3
files were dropped as bad.

Unit firing rates are low (median 0.18 to 0.63 Hz for the `hpc`, `ca3` and `ls` units). 11 units
fire above 50 Hz on average; the files carry no isolation quality or single-unit label, so all
units are kept.

In [ ]:
# Do the spikes, LFP, sleep states and trials share one clock?
# (1) Spikes against LFP: locking of spikes to the 120-200 Hz phase with the spikes shifted in time.
for key, name in (("clock_hpc", "hippocampal units, inside CA1 ripples"), ("clock_ls", "LS units, inside high-power epochs")):
    curves = [r[key] for r in R if r.get(key) is not None and np.isfinite(r[key]["r"]).all()]
    shifts = np.array(curves[0]["shifts_ms"])
    M = np.array([c["r"] for c in curves])
    best = shifts[M.argmax(axis=1)]
    print(f"{name}: {len(curves)} recordings; median resultant by shift (ms):",
          dict(zip(shifts.tolist(), np.round(np.median(M, axis=0), 3).tolist())))
    print(f"   recordings whose locking peaks within +-2 ms of zero shift: {np.sum(np.abs(best) <= 2)} of {len(best)};"
          f" peaks elsewhere: {sorted(collections.Counter(best[np.abs(best) > 2].tolist()).items())}")

# (2) Sleep states against LFP: CA1 ripples should fall in NREM far more than in WAKE.
def rate_in(times, states, label, shift=0.0):
    sel = states["label"] == label
    s0, s1 = states["start"][sel] + shift, states["stop"][sel] + shift
    order = np.argsort(s0)
    n = ls_hfo.counts_in(np.sort(times), s0[order], s1[order]).sum()
    return n / (s1 - s0).sum()

rows = []
for r in R:
    if "hpc_events" in r and r["meta"]["states"] is not None:
        st, t = r["meta"]["states"], r["hpc_events"]["peak"]
        row = {"file": r["path"].split("/")[1][:30]}
        for sh in (-600, 0, 600):
            row[sh] = rate_in(t, st, "Non-REM", sh) / max(rate_in(t, st, "Awake", sh), 1e-9)
        rows.append(row)
cl = pd.DataFrame(rows)
print(f"\nCA1 ripple rate in NREM / WAKE over {len(cl)} recordings, median: true labels {cl[0].median():.2f}, "
      f"labels shifted -10 min {cl[-600].median():.2f}, +10 min {cl[600].median():.2f}; "
      f"true labels give the largest ratio in {(cl[0] >= cl[[-600, 600]].max(axis=1)).sum()} of {len(cl)}")

# (3) Trials against states: the animal is awake on the maze, so trial time should not be scored NREM.
rows = []
for r in R:
    m = r["meta"]
    if m["states"] is not None and m["trials"] is not None:
        st = m["states"]
        nrem = st["label"] == "Non-REM"
        row = {}
        for sh in (-3600, 0, 3600):
            t = np.concatenate([np.arange(a, b, 0.1) for a, b in zip(m["trials"]["start"] + sh, m["trials"]["stop"] + sh)])
            t = t[(t > 0) & (t < m["duration"])]
            row[sh] = np.mean(ls_hfo.state_of(t, st) == "Non-REM") if len(t) else np.nan
        rows.append(row)
tc = pd.DataFrame(rows)
print(f"fraction of trial time scored NREM over {len(tc)} recordings, mean: true {tc[0].mean():.3f}, "
      f"trials shifted -1 h {tc[-3600].mean():.3f}, +1 h {tc[3600].mean():.3f}")


Three checks that the streams share one clock, each with a contrast that an offset would destroy.

1. Spikes and LFP: the phase of the 120-200 Hz band at each spike is most consistent at zero shift
   and falls off within 5 to 10 ms, about one cycle. This holds for the hippocampal units inside CA1
   ripples (32 of 39 recordings peak within +-2 ms) and for the LS units in high-power epochs (75 of
   78). The recordings that peak elsewhere have few spikes in the windows; this check supports a
   common clock in most recordings and does not establish it in those.
2. Sleep states and LFP: CA1 ripples are about 9 times as frequent in NREM as in WAKE with the
   states as stored, and about 2 times with the state table shifted by 10 min.
3. Trials and states: 1.7% of trial time is scored NREM, against 31 to 39% with the trials shifted
   by 1 h.

The data are consistent with one clock for spikes, LFP, sleep states and trials.

## dataset-composition

> LS was recorded in 6 rats, with 1989 LS neurons; 37 recordings had CA1 and LS and 25 had CA3 and
> LS recorded simultaneously; LS HFO rates come from 89 recordings and CA1 ripple rates from 53.

In [ ]:
def has_units(r, region):
    loc = r["meta"]["units"]["location"]
    return loc is not None and region in set(loc)

def has_el(r, region):
    return region in set(r["meta"]["el_location"])

animals = sorted({r["meta"]["subject"] for r in R})
n_ls_units = sum(len(ls_hfo.sorted_spikes(r, "ls")) for r in R)
n_ls_tested = sum(sum(p["n"] > 50 for p in r["phase_locking"]) for r in R)
comp = pd.DataFrame({
    "reading": ["(a) units", "(b) electrodes", "(c) electrodes and sleep states"],
    "CA1+LS (37)": [sum(has_units(r, "ls") and has_units(r, "hpc") for r in results),
                    sum(has_el(r, "ls") and has_el(r, "hpc") for r in results), None],
    "CA3+LS (25)": [sum(has_units(r, "ls") and has_units(r, "ca3") for r in results),
                    sum(has_el(r, "ls") and has_el(r, "ca3") for r in results), None],
    "LS rate recordings (89)": [sum(has_units(r, "ls") for r in results), sum(has_el(r, "ls") for r in results),
                                sum(has_el(r, "ls") and r["meta"]["states"] is not None for r in results)],
    "CA1 rate recordings (53)": [sum(has_units(r, "hpc") for r in results), sum(has_el(r, "hpc") for r in results),
                                 sum(has_el(r, "hpc") and r["meta"]["states"] is not None for r in results)],
})
print(f"animals with LS recordings: {len(animals)} {animals}")
print(f"LS neurons: (a) all units labelled ls {n_ls_units}; (b) with more than 50 spikes in high-power epochs {n_ls_tested} (paper 1989)")
print(comp.to_string(index=False))
tol = {"CA1+LS (37)": (37, 3.7), "CA3+LS (25)": (25, 2.5), "LS rate recordings (89)": (89, 8.9), "CA1 rate recordings (53)": (53, 5.3)}
within = {k: [v is not None and abs(v - c) <= t for v in comp[k]] for k, (c, t) in tol.items()}
print("within tolerance by reading:", within)
print("LS neurons within tolerance:", {"(a)": abs(n_ls_units - 1989) <= 198.9, "(b)": abs(n_ls_tested - 1989) <= 198.9})


**Result: partially reproduced.** The animal count matches (6). Three of the five other counts are
within tolerance under one reading: LS neurons with more than 50 spikes in high-power epochs
(2046 against 1989), files with LS and `hpc` units (37 against 37) and files with LS electrodes
(94 against 89). Files with LS and CA3 units number 10 against 25, and no reading gives 53 CA1
recordings (40, 72 or 43). All units labelled `ls` number 2448.

## fig1c-hfo-each-animal

> Transient 120-180 Hz HFOs occur in LS in each of the 6 animals, with a robust increase in the
> firing probability of LS neurons at the events.

Spike probability is the number of spikes in +-10 ms of the event time, per event and neuron,
against as many uniformly random times in the same recording.

In [ ]:
# fig1c-hfo-each-animal: frequency and HFO-triggered LS firing by animal
rng = np.random.default_rng(102)
rows, curves = [], {}
for subj in animals:
    freqs, ev_p, rnd_p, n_events, peth_sum, peth_rnd = [], [], [], 0, 0, 0
    for r in [r for r in R if r["meta"]["subject"] == subj]:
        e = r["ls_events"]
        freqs.append(e["frequency"])
        n_events += len(e["peak"])
        units = ls_hfo.sorted_spikes(r, "ls")
        if len(e["peak"]) == 0 or not units:
            continue
        rnd_t = rng.uniform(0.2, r["meta"]["duration"] - 0.2, len(e["peak"]))
        for s in units.values():
            pe, pr = ls_hfo.peth(s, e["peak"]), ls_hfo.peth(s, rnd_t)
            ev_p.append(pe[90:111].sum())   # +-10 ms around the event time (1 ms bins)
            rnd_p.append(pr[90:111].sum())
            peth_sum = peth_sum + pe
            peth_rnd = peth_rnd + pr
    d = np.array(ev_p) - np.array(rnd_p)
    f = np.concatenate(freqs)
    curves[subj] = (peth_sum / len(ev_p), peth_rnd / len(ev_p))
    rows.append({"animal": subj, "HFOs": n_events, "median freq (Hz)": round(np.nanmedian(f), 1),
                 "LS neurons": len(ev_p), "P(spike) +-10 ms at HFO": round(np.mean(ev_p), 4),
                 "at random times": round(np.mean(rnd_p), 4), "difference": round(d.mean(), 4),
                 "SEM": round(d.std(ddof=1) / np.sqrt(len(d)), 4), "difference / SEM": round(d.mean() / (d.std(ddof=1) / np.sqrt(len(d))), 1)})
f1c = pd.DataFrame(rows)
f1c["freq in 120-180"] = f1c["median freq (Hz)"].between(120, 180)
f1c["above 3 SEM"] = f1c["difference / SEM"] > 3
print(f1c.to_string(index=False))
print("animals meeting both:", int((f1c["freq in 120-180"] & f1c["above 3 SEM"]).sum()), "of", len(f1c))
fig, axes = plt.subplots(1, len(animals), figsize=(15, 2.6), sharey=False)
lag = np.arange(-99.5, 100, 1)
for ax, subj in zip(axes, animals):
    ax.plot(lag, curves[subj][0], "k", lw=1, label="HFO")
    ax.plot(lag, curves[subj][1], "r", lw=1, label="random")
    ax.set_title(subj); ax.set_xlabel("ms from HFO")
axes[0].set_ylabel("spikes per event per 1 ms bin"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("plotted range (spikes per event per bin):", {s: (round(float(min(c[0].min(), c[1].min())), 4), round(float(max(c[0].max(), c[1].max())), 4)) for s, c in curves.items()})


Variants that do not change the outcome: only the events in NREM, and only the events with a CA1
ripple within 25 ms (animals without states or hippocampal channels drop out).

In [ ]:
# Variant (does not change the outcome): NREM events only, and events with a CA1 ripple within 25 ms.
rng = np.random.default_rng(1021)
def triggered(sel_fn):
    out = {}
    for subj in animals:
        d = []
        for r in [r for r in R if r["meta"]["subject"] == subj]:
            e = r["ls_events"]
            t = e["peak"][sel_fn(r, e)]
            units = ls_hfo.sorted_spikes(r, "ls")
            if len(t) < 5 or not units:
                continue
            rnd_t = rng.uniform(0.2, r["meta"]["duration"] - 0.2, len(t))
            d += [ls_hfo.peth(s, t)[90:111].sum() - ls_hfo.peth(s, rnd_t)[90:111].sum() for s in units.values()]
        d = np.array(d)
        out[subj] = (len(d), round(d.mean(), 4) if len(d) else None, round(d.mean() / (d.std(ddof=1) / np.sqrt(len(d))), 1) if len(d) > 2 else None)
    return out
def coupled(r, e):
    if "hpc_events" not in r:
        return np.zeros(len(e["peak"]), bool)
    rp = np.sort(r["hpc_events"]["peak"])
    k = np.clip(np.searchsorted(rp, e["peak"]), 1, max(len(rp) - 1, 1))
    near = np.minimum(np.abs(e["peak"] - rp[k - 1]), np.abs(e["peak"] - rp[np.minimum(k, len(rp) - 1)])) if len(rp) else np.full(len(e["peak"]), np.inf)
    return near <= 0.025
print("NREM events only (neurons, mean difference, difference/SEM):", triggered(lambda r, e: e["state"] == "Non-REM"))
print("events with a CA1 ripple within 25 ms:", triggered(coupled))


**Result: not reproduced.** Every animal's detected events have a median frequency of 152 to 158
Hz, inside 120 to 180 Hz, but LS firing at the events exceeds firing at random times by more than 3
SEM in DT1 only (6.1 SEM). In DT2 and DT5 it is higher by 1.2 and 1.3 SEM, and in DT7, DT8 and DT9
it is lower. With NREM events only, DT2 (3.3 SEM) and DT9 (3.6 SEM) pass; with ripple-coupled
events, DT9 passes (4.2 SEM). Many detected events do not behave as LS population bursts.

## fig1c-event-count

> Across 6 animals, 29,955 LS HFO events were analysed.

In [ ]:
# fig1c-event-count
n_all = sum(len(r["ls_events"]["peak"]) for r in R)
n_units = sum(len(r["ls_events"]["peak"]) for r in R if has_units(r, "ls"))
print(f"LS HFOs: (a) all {len(R)} recordings with LS electrodes {n_all}; (b) the {sum(has_units(r, 'ls') for r in R)} recordings with LS units {n_units}  (paper 29,955, tolerance 2,995.5)")
print("by animal (a):", {s: sum(len(r["ls_events"]["peak"]) for r in R if r["meta"]["subject"] == s) for s in animals})
by_state = collections.Counter(np.concatenate([r["ls_events"]["state"] for r in R]))
print("by sleep state at the event (all recordings; '' = no states or unscored):", dict(by_state))


**Result: reproduced.** In the recordings with LS units, 31,984 events are detected, within 10% of
29,955. Over all recordings with LS electrodes there are 67,186, 38,008 of them from DT7, and more
fall in WAKE (27,676) than in NREM (20,019). The match of reading (b) does not mean the same events
were found.

## figs1-participation

> On average 16% of LS neurons take part in a given HFO, firing an average of 0.23 spikes per event.

In [ ]:
# figs1-participation
frac_ev, spk_ev, per_rec = [], [], []
for r in R:
    units = ls_hfo.sorted_spikes(r, "ls")
    e = r["ls_events"]
    if not units or len(e["peak"]) == 0:
        continue
    C = np.array([ls_hfo.counts_in(s, e["start"], e["stop"]) for s in units.values()])  # neurons x events
    frac_ev.append((C > 0).mean(axis=0))
    spk_ev.append(C.ravel())
    per_rec.append({"file": r["path"].split("/")[1][:30], "neurons": len(units), "events": C.shape[1],
                    "fraction": (C > 0).mean(), "spikes": C.mean()})
per_rec = pd.DataFrame(per_rec)
fa, sa = np.concatenate(frac_ev).mean(), np.concatenate(spk_ev).mean()
print(f"(a) pooled over {sum(len(x) for x in frac_ev)} events: fraction of LS neurons firing {fa:.3f}, spikes per neuron per event {sa:.3f}")
print(f"(b) mean of {len(per_rec)} recordings: fraction {per_rec.fraction.mean():.3f}, spikes {per_rec.spikes.mean():.3f}")
print("paper: 0.16 (tolerance 0.016) and 0.23 (0.023)")
print(f"per-recording fraction: median {per_rec.fraction.median():.3f}, range {per_rec.fraction.min():.3f} to {per_rec.fraction.max():.3f}")


**Result: not reproduced.** Pooled over events, 2.7% of LS neurons fire in an HFO, with 0.051
spikes per neuron per event; the mean over recordings gives 3.5% and 0.068. Both are a fifth to a
third of the paper's values. The highest recording reaches 18.2%.

## fig1d-phase-locking

> The vast majority of LS neurons (1840 of 1989, 92%) are significantly phase-locked to the
> 120-200 Hz band (Rayleigh P < 0.05), with a strong preference for the trough of each cycle.

The phase is taken on each recording's HFO detection channel; the paper does not say which channel
it used.

In [ ]:
# fig1d-phase-locking
pl = pd.DataFrame([{"animal": r["meta"]["subject"], **{k: p[k] for k in ("n", "p", "r", "phase")},
                    "r_null": np.nanmean(p["r_null"])} for r in R for p in r["phase_locking"]])
tested = pl[pl.n > 50]
sig = tested[tested.p < 0.05]
print(f"LS neurons: {len(pl)}; with more than 50 spikes in high-power epochs: {len(tested)}; significant (Rayleigh p < 0.05): {len(sig)}")
print(f"fraction significant: (a) of tested {len(sig) / len(tested):.3f}; (b) of all LS neurons {len(sig) / len(pl):.3f}  (paper 0.925, tolerance 0.0925)")
print(f"resultant length of tested neurons: median {tested.r.median():.3f}; with the phase circularly shifted: median {tested.r_null.median():.3f}")
mean_dir = np.angle(np.exp(1j * sig.phase).mean())
rt = len(sig) * np.abs(np.exp(1j * sig.phase).mean())
p_pref = np.exp(np.sqrt(1 + 4 * len(sig) + 4 * (len(sig) ** 2 - rt ** 2)) - (1 + 2 * len(sig)))
print(f"preferred phases of significant neurons: circular mean {np.degrees(mean_dir) % 360:.1f} deg (trough = 180),"
      f" resultant {np.abs(np.exp(1j * sig.phase).mean()):.3f}, Rayleigh p = {p_pref:.2g}")
print(f"within 45 deg of the trough: {abs(abs(np.degrees(mean_dir)) - 180) <= 45}")
print("by animal:")
print(tested.groupby("animal").apply(lambda d: pd.Series({"tested": len(d), "significant": int((d.p < 0.05).sum()),
      "fraction": round((d.p < 0.05).mean(), 3), "median r": round(d.r.median(), 3),
      "mean phase (deg)": round(np.degrees(np.angle(np.exp(1j * d.phase[d.p < 0.05]).mean())) % 360, 1)})).to_string())
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(tested.r, bins=50, range=(0, 0.6), color="k", histtype="step", label="data")
axes[0].hist(tested.r_null, bins=50, range=(0, 0.6), color="r", histtype="step", label="shifted phase")
axes[0].set_xlabel("resultant length"); axes[0].legend()
axes[1].hist(np.degrees(sig.phase) % 360, bins=36, range=(0, 360), color="k")
axes[1].set_xlabel("preferred phase (deg; 180 = trough)")
plt.tight_layout(); plt.show()
print(f"resultant lengths above 0.6 (outside the axis): data {(tested.r > 0.6).sum()}, shifted {(tested.r_null > 0.6).sum()}")


**Result: partially reproduced.** 1151 of 2046 tested LS neurons (56.3%) are significantly
phase-locked, 47.0% of all 2448; the paper reports 92.5%. The trough preference holds: the
preferred phases of significant neurons centre on 189 degrees (resultant 0.755; the Rayleigh p
underflows to 0). Fractions range from 5.7% (DT7) to 70.4% (DT5) by animal. Median resultant lengths
are 0.124 for the data and 0.048 with the phase circularly shifted.

## fig1f-shorter-than-ripples and fig1g-slower-than-ripples

> LS HFOs are shorter than CA1 ripples from the same animals (27 +- 9 against 31 +- 14 ms;
> rank-sum P < 0.01).

> LS HFOs are slower in frequency than CA1 ripples (147 +- 5.7 against 159 +- 8 Hz; rank-sum
> P < 0.01).

In [ ]:
# fig1f-shorter-than-ripples and fig1g-slower-than-ripples: animals with both LS and hippocampal channels
both = [r for r in R if "hpc_events" in r]
ls_d = np.concatenate([r["ls_events"]["duration"] for r in both]) * 1000
ca_d = np.concatenate([r["hpc_events"]["duration"] for r in both]) * 1000
ls_f = np.concatenate([r["ls_events"]["frequency"] for r in both])
ca_f = np.concatenate([r["hpc_events"]["frequency"] for r in both])
ls_f, ca_f = ls_f[np.isfinite(ls_f)], ca_f[np.isfinite(ca_f)]
print(f"{len(both)} recordings with both channels, animals {sorted({r['meta']['subject'] for r in both})}")
for name, a, b, unit, ref in (("duration", ls_d, ca_d, "ms", "27 +- 9 against 31 +- 14"), ("frequency", ls_f, ca_f, "Hz", "147 +- 5.7 against 159 +- 8")):
    p = stats.mannwhitneyu(a, b).pvalue
    print(f"{name}: LS {a.mean():.1f} +- {a.std():.1f} {unit} (median {np.median(a):.1f}, n = {len(a)}); "
          f"CA1 {b.mean():.1f} +- {b.std():.1f} (median {np.median(b):.1f}, n = {len(b)}); rank-sum p = {p:.2g}  [paper {ref}]")
print("variant, all recordings with LS electrodes:",
      f"LS duration {np.concatenate([r['ls_events']['duration'] for r in R]).mean() * 1000:.1f} ms,",
      f"frequency {np.nanmean(np.concatenate([r['ls_events']['frequency'] for r in R])):.1f} Hz")
rows = []
for subj in sorted({r["meta"]["subject"] for r in both}):
    rr = [r for r in both if r["meta"]["subject"] == subj]
    a = np.concatenate([r["ls_events"]["duration"] for r in rr]) * 1000
    b = np.concatenate([r["hpc_events"]["duration"] for r in rr]) * 1000
    fa = np.concatenate([r["ls_events"]["frequency"] for r in rr])
    fb = np.concatenate([r["hpc_events"]["frequency"] for r in rr])
    rows.append({"animal": subj, "LS dur": round(a.mean(), 1), "CA1 dur": round(b.mean(), 1),
                 "p dur": stats.mannwhitneyu(a, b).pvalue, "LS freq": round(np.nanmean(fa), 1),
                 "CA1 freq": round(np.nanmean(fb), 1), "p freq": stats.mannwhitneyu(fa[np.isfinite(fa)], fb[np.isfinite(fb)]).pvalue})
print(pd.DataFrame(rows).to_string(index=False))
# NREM-only variant
st_ls = np.concatenate([r["ls_events"]["state"] for r in both]); st_ca = np.concatenate([r["hpc_events"]["state"] for r in both])
a, b = ls_d[st_ls == "Non-REM"], ca_d[st_ca == "Non-REM"]
fa = np.concatenate([r["ls_events"]["frequency"] for r in both])[st_ls == "Non-REM"]; fb = np.concatenate([r["hpc_events"]["frequency"] for r in both])[st_ca == "Non-REM"]
print(f"variant, NREM events only: duration LS {a.mean():.1f} +- {a.std():.1f}, CA1 {b.mean():.1f} +- {b.std():.1f} ms (p = {stats.mannwhitneyu(a, b).pvalue:.2g});"
      f" frequency LS {np.nanmean(fa):.1f} +- {np.nanstd(fa):.1f}, CA1 {np.nanmean(fb):.1f} +- {np.nanstd(fb):.1f} Hz (p = {stats.mannwhitneyu(fa[np.isfinite(fa)], fb[np.isfinite(fb)]).pvalue:.2g})")
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(ca_d, bins=60, range=(0, 150), density=True, histtype="step", color="k", label="CA1")
axes[0].hist(ls_d, bins=60, range=(0, 150), density=True, histtype="step", color="m", label="LS")
axes[0].set_xlabel("duration (ms)"); axes[0].legend()
axes[1].hist(ca_f, bins=60, range=(100, 220), density=True, histtype="step", color="k")
axes[1].hist(ls_f, bins=60, range=(100, 220), density=True, histtype="step", color="m")
axes[1].set_xlabel("peak frequency (Hz)")
plt.tight_layout(); plt.show()
print(f"outside the axes: durations above 150 ms LS {(ls_d > 150).sum()}, CA1 {(ca_d > 150).sum()};"
      f" frequencies outside 100-220 Hz LS {((ls_f < 100) | (ls_f > 220)).sum()}, CA1 {((ca_f < 100) | (ca_f > 220)).sum()}")


**Result: fig1f-shorter-than-ripples reproduced.** LS HFOs last 33.9 +- 25.0 ms and CA1 ripples
38.3 +- 23.7 ms, both inside the tolerances of the paper's SDs, and LS HFOs are shorter (medians
23.2 and 32.0 ms; the rank-sum p underflows to 0). The means are 6 to 7 ms above the paper's. In four of
five animals LS HFOs are significantly shorter; in DT8 they are not (27.5 against 28.0 ms, p =
0.83). With NREM events only, the means are 38.0 and 38.4 ms.

**Result: fig1g-slower-than-ripples not reproduced.** LS HFOs have a mean frequency of 153.6 +- 8.3
Hz and CA1 ripples 150.8 +- 9.2 Hz: the LS events are faster, the reverse of the claim, and both
means are outside the tolerances (147 +- 5.7 and 159 +- 8). LS events are faster in four of five
animals. With NREM events only, LS are slower by 0.8 Hz (150.3 against 151.1 Hz).

## fig1h-no-amplitude-frequency-correlation

> LS HFOs show no significant correlation between event amplitude and peak frequency.

In [ ]:
# fig1h-no-amplitude-frequency-correlation
amp = np.concatenate([r["ls_events"]["amplitude"] for r in R])
frq = np.concatenate([r["ls_events"]["frequency"] for r in R])
ok = np.isfinite(frq)
pr = stats.pearsonr(amp[ok], frq[ok])
sr = stats.spearmanr(amp[ok], frq[ok])
print(f"all {ok.sum()} LS HFOs: Pearson r = {pr.statistic:.3f}, p = {pr.pvalue:.2g}; Spearman rho = {sr.statistic:.3f}, p = {sr.pvalue:.2g}")
rows = []
for subj in animals:
    a = np.concatenate([r["ls_events"]["amplitude"] for r in R if r["meta"]["subject"] == subj])
    f = np.concatenate([r["ls_events"]["frequency"] for r in R if r["meta"]["subject"] == subj])
    k = np.isfinite(f)
    t = stats.pearsonr(a[k], f[k])
    rows.append({"animal": subj, "events": int(k.sum()), "r": round(t.statistic, 3), "p": t.pvalue})
print(pd.DataFrame(rows).to_string(index=False))
st = np.concatenate([r["ls_events"]["state"] for r in R])
k = ok & (st == "Non-REM")
t = stats.pearsonr(amp[k], frq[k])
print(f"variant, NREM events only ({k.sum()}): r = {t.statistic:.3f}, p = {t.pvalue:.2g}")
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(frq[ok], amp[ok], ".", ms=1, alpha=0.3, color="m")
ax.set_xlim(100, 220); ax.set_ylim(4, 30); ax.set_xlabel("peak frequency (Hz)"); ax.set_ylabel("amplitude (SD)")
plt.tight_layout(); plt.show()
print(f"points outside the axes: {((frq[ok] < 100) | (frq[ok] > 220) | (amp[ok] > 30)).sum()} of {ok.sum()}")


**Result: not reproduced.** Over 67,186 events the Pearson correlation is small but significant (r =
0.056, p = 2.4e-47; Spearman rho = 0.123). By animal, r is -0.009 to 0.080, significant in four of six;
with NREM events only, r = 0.028 (p = 7.3e-05). The effect is weak, and a test over tens of
thousands of events finds it; the paper does not name its test.

## fig2-confined-to-ls

> HFO events are seen only at recording sites within LS, not at sites in neighbouring structures.

DT7 has probes that span cingulate cortex (`acg`), corpus callosum (`cc`), medial septum (`ms`) and
fimbria (`sFi`) with LS. For each channel, the 120-200 Hz power within +-20 ms of the LS HFO times in
up to 30 min of all channels, divided by the power at random times.

In [ ]:
# fig2-confined-to-ls
rows = []
for r in R:
    if "localization" not in r:
        continue
    L, loc = r["localization"], r["meta"]["el_location"]
    good = r["channel_stats"]["good"]
    row = {"file": r["path"].split("/")[1][:34], "HFOs in block": L["n_events"],
           "block (min)": round((L["window"][1] - L["window"][0]) / 60, 1)}
    for g in sorted(set(loc)):
        row[g] = round(float(np.nanmedian(L["ratio"][(loc == g) & good])), 2)
    rows.append(row)
loc_df = pd.DataFrame(rows)
print("median over channels of HFO-triggered / random 120-200 Hz power, by region:")
print(loc_df.to_string(index=False))
others = [c for c in loc_df.columns if c not in ("file", "HFOs in block", "block (min)", "ls", "hpc", "ca3", "unknown")]
used = loc_df[loc_df["HFOs in block"] >= 20]
win = used.apply(lambda row: all(row["ls"] > row[c] for c in others if pd.notna(row[c])), axis=1)
print(f"recordings with at least 20 HFOs in the block: {len(used)}; LS above every other non-hippocampal region in {int(win.sum())}")


**Result: not reproduced.** Five recordings have at least 20 HFOs in the block. LS channels have the
highest ratio in two: DT7 2017-03-18 (5.38 against 5.31 for cingulate cortex) and 2017-03-21 (8.25
against 8.05 for medial septum). In the other three, medial septum (6.05 against 5.18), corpus
callosum (50.1 against 42.0) or fimbria (72.1 against 64.8) channels are higher. The detected events
reach channels outside LS at similar or larger relative power.

## LS pairs: fig4b-pre-post-unchanged, fig4b-hfo-above-behav, fig4d-compression

> The co-activation of LS neuron pairs during HFOs (excess correlation) is not modulated by waking
> experience on a familiar maze (PRE against POST, rank-sum P = 0.36).

> Excess correlations during HFOs (PRE and POST) are higher than during maze traversal (rank-sum
> P < 0.001), and those of PRE and POST HFOs exceed chance from spikes jittered by +-200 ms (P < 0.01).

> Cross-correlograms of LS pairs during HFOs correlate with those during traversal above chance up
> to about 20-fold temporal compression (P < 0.01), similarly for PRE and POST (P > 0.01 for all
> compression factors).

The pair selection of the criteria (pairs on different shanks with more than 5 spike pairs within
+-200 ms in each epoch) is applied first; it decides which of these claims have enough data.

In [ ]:
# Pairs of LS neurons on different shanks: CCGs of spikes inside LS HFOs (PRE, POST) and inside trials (BEHAV).
rng = np.random.default_rng(109)
FACTORS = [1, 2, 4, 6, 8, 10, 15, 20, 30, 40, 50]
NB = 400                       # bins of 1 ms on each side
W200 = slice(NB - 200, NB + 201)

def warp_corr(c_hfo, c_beh, factor):
    """Correlation of the BEHAV CCG over +-120 ms with the HFO CCG over +-120/factor ms,
    interpolated to the same 241 points."""
    lags = np.arange(-NB, NB + 1)
    x = np.linspace(-120 / factor, 120 / factor, 241)
    a = np.interp(x, lags, c_hfo)
    b = c_beh[NB - 120:NB + 121]
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return np.corrcoef(a, b)[0, 1]

pair_rows, comp_rows, kept_sessions = [], [], 0
t_start = time.time()
for r in R:
    m = r["meta"]
    units = ls_hfo.sorted_spikes(r, "ls")
    if m["epochs"] is None or m["trials"] is None or len(units) < 2 or m["units"]["group"] is None:
        continue
    e = r["ls_events"]
    order = np.argsort(e["start"])
    es, ee, ep = e["start"][order], e["stop"][order], e["peak"][order]
    pre = ep < m["epochs"]["stop"][0]
    post = ep >= m["epochs"]["start"][2]
    to = np.argsort(m["trials"]["start"])
    ts, tt = m["trials"]["start"][to], m["trials"]["stop"][to]
    sp = {"PRE": {}, "BEHAV": {}, "POST": {}}
    for i, s in units.items():
        sp["PRE"][i] = ls_hfo.in_intervals(s, es[pre], ee[pre])
        sp["POST"][i] = ls_hfo.in_intervals(s, es[post], ee[post])
        sp["BEHAV"][i] = ls_hfo.in_intervals(s, ts, tt)
    grp = m["units"]["group"]
    ids = list(units)
    n_pairs = 0
    for a_ in range(len(ids)):
        for b_ in range(a_ + 1, len(ids)):
            i, j = ids[a_], ids[b_]
            if grp[i] == grp[j]:
                continue
            C = {k: ls_hfo.ccg(sp[k][i], sp[k][j], half=NB / 1000) for k in sp}
            if any(C[k][W200].sum() <= 5 for k in C):
                continue
            row = {"file": r["path"].split("/")[1][:30], "subject": m["subject"]}
            for k in C:
                cn = C[k] / np.sqrt(len(sp[k][i]) * len(sp[k][j]))
                row[k] = ls_hfo.excess_correlation(cn)
                ji = np.sort(sp[k][i] + rng.uniform(-0.2, 0.2, len(sp[k][i])))
                jj = np.sort(sp[k][j] + rng.uniform(-0.2, 0.2, len(sp[k][j])))
                row[k + "_jitter"] = ls_hfo.excess_correlation(ls_hfo.ccg(ji, jj, half=NB / 1000) / np.sqrt(len(ji) * len(jj)))
                row["bias_" + k] = cn[NB + 1:NB + 201].sum() - cn[NB - 200:NB].sum()
                C[k] = cn
            for k in ("PRE", "POST"):
                shift = rng.integers(1, 2 * NB)
                for fct in FACTORS:
                    comp_rows.append({"file": row["file"], "epoch": k, "factor": fct,
                                      "corr": warp_corr(C[k], C["BEHAV"], fct),
                                      "chance": warp_corr(np.roll(C[k], shift), C["BEHAV"], fct)})
            pair_rows.append(row)
            n_pairs += 1
    kept_sessions += n_pairs > 0
pairs = pd.DataFrame(pair_rows)
comp = pd.DataFrame(comp_rows)
print(f"pairs kept: {len(pairs)} in {kept_sessions} recordings (paper 7,172 pairs in 53 sessions); {time.time() - t_start:.0f} s")
print("pairs by animal:", pairs.subject.value_counts().sort_index().to_dict())
print(pairs[["PRE", "BEHAV", "POST", "PRE_jitter", "BEHAV_jitter", "POST_jitter"]].describe().round(4).to_string())


Only 106 pairs from 10 recordings in 3 animals pass the selection, against 7,172 pairs in 53
sessions in the paper. Few LS spikes fall inside the detected HFOs (see figs1-participation), so
most pairs have 5 or fewer spike pairs within +-200 ms in PRE or POST. The pair-level tests below
use these 106 pairs. The per-session claims are blocked: 7 recordings have at least 5 pairs
(fig4-sessions-few-change) and 4 have at least 10 (fig4c-temporal-bias); their outputs are printed
for the record.

In [ ]:
# fig4b-pre-post-unchanged and fig4b-hfo-above-behav
t = stats.mannwhitneyu(pairs.PRE, pairs.POST)
print(f"PRE against POST (rank-sum): medians {pairs.PRE.median():.4f} and {pairs.POST.median():.4f}, p = {t.pvalue:.3g};"
      f" median of POST - PRE over pairs {np.median(pairs.POST - pairs.PRE):.4f}; Wilcoxon signed-rank p = {stats.wilcoxon(pairs.PRE, pairs.POST).pvalue:.3g} (variant)")
for k in ("PRE", "POST"):
    t = stats.mannwhitneyu(pairs[k], pairs.BEHAV)
    print(f"{k} against BEHAV: medians {pairs[k].median():.4f} and {pairs.BEHAV.median():.4f}, rank-sum p = {t.pvalue:.3g}")
for k in ("PRE", "BEHAV", "POST"):
    t = stats.mannwhitneyu(pairs[k], pairs[k + "_jitter"])
    print(f"{k} against jittered: medians {pairs[k].median():.4f} and {pairs[k + '_jitter'].median():.4f}, rank-sum p = {t.pvalue:.3g}")
print("by animal (median excess correlation):")
print(pairs.groupby("subject")[["PRE", "BEHAV", "POST"]].median().round(4).to_string())
print("by animal, PRE against POST rank-sum p:", pairs.groupby("subject").apply(lambda d: round(stats.mannwhitneyu(d.PRE, d.POST).pvalue, 3)).to_dict())
fig, ax = plt.subplots(figsize=(5, 3))
for k, c in (("PRE", "g"), ("BEHAV", "k"), ("POST", "b")):
    ax.hist(pairs[k], bins=60, range=(-0.05, 0.25), histtype="step", color=c, label=k)
ax.set_xlabel("excess correlation"); ax.legend(); plt.tight_layout(); plt.show()
print("values outside the axis:", {k: int(((pairs[k] < -0.05) | (pairs[k] > 0.25)).sum()) for k in ("PRE", "BEHAV", "POST")})


**Result: fig4b-pre-post-unchanged reproduced.** PRE and POST excess correlations do not differ
(medians 0.0062 and 0.0059, rank-sum p = 0.81), on 106 pairs against the paper's 7,172.

**Result: fig4b-hfo-above-behav reproduced.** HFO excess correlations are higher than BEHAV (median
0.0003; PRE p = 9.1e-24, POST p = 3.8e-25), and above their jittered values (p = 3.6e-31 and
7.8e-35); BEHAV is also slightly above its jitter (p = 0.0022), as in the paper. Same 106 pairs.

In [ ]:
# fig4-sessions-few-change
rows = []
for f, d in pairs.groupby("file"):
    if len(d) >= 5:
        p = stats.mannwhitneyu(d.PRE, d.POST).pvalue
        rows.append({"file": f, "pairs": len(d), "p": p, "direction": "PRE < POST" if d.PRE.median() < d.POST.median() else "PRE > POST"})
sess = pd.DataFrame(rows)
sig = sess[sess.p < 0.01]
print(f"recordings with at least 5 pairs: {len(sess)}; PRE differs from POST (p < 0.01): {len(sig)} ({len(sig) / len(sess):.3f}; paper 7 of 53 = 0.132)")
print("directions among them:", sig.direction.value_counts().to_dict())
print(sess.sort_values("p").head(12).to_string(index=False))


In [ ]:
# fig4c-temporal-bias
rows = []
for f, d in pairs.groupby("file"):
    if len(d) >= 10:
        rows.append({"file": f, "pairs": len(d), "pre_behav": stats.pearsonr(d.bias_PRE, d.bias_BEHAV).statistic,
                     "behav_post": stats.pearsonr(d.bias_BEHAV, d.bias_POST).statistic})
tb = pd.DataFrame(rows)
t = stats.ttest_ind(tb.pre_behav, tb.behav_post)
print(f"{len(tb)} recordings with at least 10 pairs; correlation of biases PRE-BEHAV mean {tb.pre_behav.mean():.3f}, BEHAV-POST mean {tb.behav_post.mean():.3f}")
print(f"two-sample t-test p = {t.pvalue:.3g}; paired t-test p = {stats.ttest_rel(tb.pre_behav, tb.behav_post).pvalue:.3g} (variant)")
print(f"one-sample t-tests against 0: PRE-BEHAV p = {stats.ttest_1samp(tb.pre_behav, 0).pvalue:.3g}, BEHAV-POST p = {stats.ttest_1samp(tb.behav_post, 0).pvalue:.3g}")
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.plot(tb.pre_behav, tb.behav_post, "k.")
ax.plot([-1, 1], [-1, 1], ":", color="grey"); ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
ax.set_xlabel("r(PRE, BEHAV)"); ax.set_ylabel("r(BEHAV, POST)"); plt.tight_layout(); plt.show()


In [ ]:
# fig4d-compression
rows = []
for (k, fct), d in comp.groupby(["epoch", "factor"]):
    d = d.dropna()
    rows.append({"epoch": k, "factor": fct, "pairs": len(d), "mean corr": round(d["corr"].mean(), 4),
                 "mean chance": round(d.chance.mean(), 4), "p vs chance": stats.wilcoxon(d["corr"], d.chance).pvalue})
cr = pd.DataFrame(rows)
pp = []
for fct in FACTORS:
    a = comp[(comp.epoch == "PRE") & (comp.factor == fct)]["corr"].dropna()
    b = comp[(comp.epoch == "POST") & (comp.factor == fct)]["corr"].dropna()
    pp.append({"factor": fct, "PRE vs POST p": stats.mannwhitneyu(a, b).pvalue})
cr = cr.merge(pd.DataFrame(pp), on="factor")
print(cr.to_string(index=False))
above = {k: [f for f in FACTORS if cr[(cr.epoch == k) & (cr.factor == f)]["p vs chance"].iloc[0] < 0.01
             and cr[(cr.epoch == k) & (cr.factor == f)]["mean corr"].iloc[0] > cr[(cr.epoch == k) & (cr.factor == f)]["mean chance"].iloc[0]] for k in ("PRE", "POST")}
print("factors where the correlation exceeds chance (p < 0.01):", above)
print("factors where PRE and POST differ (p <= 0.01):", [f for f, p in zip(FACTORS, cr.drop_duplicates("factor")["PRE vs POST p"]) if p <= 0.01])


**Result: fig4d-compression not reproduced.** PRE correlations exceed chance at factors 1 and 2 only,
POST at 1, 2, 4 and 20. PRE and POST do not differ at any factor (smallest p = 0.038). The number of
pairs with a defined correlation falls from 59 to 77 at factor 1 to 5 to 11 at factor 50, as the
HFO correlogram is empty in the narrow windows.

## figs2-post-rate-increase

> Firing rates increase during post-behavior NREM sleep in CA1, CA3 and LS.

In [ ]:
# figs2-post-rate-increase
rows = []
for r in R + skipped:
    m = r["meta"]
    if m["states"] is None or m["epochs"] is None or m["units"]["location"] is None:
        continue
    st = m["states"]
    nrem = st["label"] == "Non-REM"
    def nrem_within(a, b):
        s0, s1 = np.clip(st["start"][nrem], a, b), np.clip(st["stop"][nrem], a, b)
        keep = s1 > s0
        return s0[keep], s1[keep]
    pre_iv = nrem_within(m["epochs"]["start"][0], m["epochs"]["stop"][0])
    post_iv = nrem_within(m["epochs"]["start"][2], m["epochs"]["stop"][2])
    dpre, dpost = (pre_iv[1] - pre_iv[0]).sum(), (post_iv[1] - post_iv[0]).sum()
    if dpre < 60 or dpost < 60:
        continue
    o1, o2 = np.argsort(pre_iv[0]), np.argsort(post_iv[0])
    for i, s in enumerate(m["units"]["spikes"]):
        s = np.sort(s)
        rows.append({"region": m["units"]["location"][i], "subject": m["subject"],
                     "pre": ls_hfo.counts_in(s, pre_iv[0][o1], pre_iv[1][o1]).sum() / dpre,
                     "post": ls_hfo.counts_in(s, post_iv[0][o2], post_iv[1][o2]).sum() / dpost})
rates_pp = pd.DataFrame(rows)
out = []
for g in ("hpc", "ca3", "ls"):
    d = rates_pp[(rates_pp.region == g) & ((rates_pp.pre > 0) | (rates_pp.post > 0))]
    w = stats.wilcoxon(d.post, d.pre)
    out.append({"region": g, "units": len(d), "recordings": None, "median PRE (Hz)": round(d.pre.median(), 3),
                "median POST (Hz)": round(d.post.median(), 3),
                "median POST/PRE": round(np.median((d.post + 1e-3) / (d.pre + 1e-3)), 3), "Wilcoxon p": w.pvalue})
print(pd.DataFrame(out).drop(columns="recordings").to_string(index=False))
print("by animal, median POST/PRE:")
print(rates_pp[(rates_pp.pre > 0) | (rates_pp.post > 0)].assign(ratio=lambda d: (d.post + 1e-3) / (d.pre + 1e-3))
      .groupby(["region", "subject"]).ratio.agg(["count", "median"]).round(3).to_string())


**Result: partially reproduced.** LS units fire more in POST NREM than in PRE NREM (median ratio
1.06, Wilcoxon p = 8.4e-07, 1326 units). The `hpc` units (ratio 1.00, p = 0.52) and the `ca3` units
(ratio 1.03, p = 0.22) do not.

## fig5b-nrem-and-rates

> LS HFOs occur primarily during NREM sleep, and their rates are lower than those of hippocampal
> SPW-Rs.

In [ ]:
# fig5b-nrem-and-rates
rows = []
for r in R:
    st = r["meta"]["states"]
    if st is None:
        continue
    row = {"file": r["path"].split("/")[1][:30], "subject": r["meta"]["subject"],
           "LS NREM": rate_in(r["ls_events"]["peak"], st, "Non-REM"), "LS WAKE": rate_in(r["ls_events"]["peak"], st, "Awake")}
    if "hpc_events" in r:
        row["CA1 NREM"] = rate_in(r["hpc_events"]["peak"], st, "Non-REM")
        row["CA1 WAKE"] = rate_in(r["hpc_events"]["peak"], st, "Awake")
    rows.append(row)
rt = pd.DataFrame(rows)
no_state = rt[rt[["LS NREM", "LS WAKE"]].isna().any(axis=1)]
print(f"recordings without NREM or without WAKE time, left out: {len(no_state)} {no_state.file.tolist()}")
rt = rt.dropna(subset=["LS NREM", "LS WAKE"])
w1 = stats.wilcoxon(rt["LS NREM"], rt["LS WAKE"])
print(f"{len(rt)} recordings with states. LS HFO rate (events/s): NREM median {rt['LS NREM'].median():.3f}, WAKE median {rt['LS WAKE'].median():.3f};"
      f" NREM > WAKE in {(rt['LS NREM'] > rt['LS WAKE']).sum()} recordings; NREM < WAKE in {(rt['LS NREM'] < rt['LS WAKE']).sum()}; median NREM - WAKE {np.median(rt['LS NREM'] - rt['LS WAKE']):.4f}; Wilcoxon p = {w1.pvalue:.3g}")
b = rt.dropna(subset=["CA1 NREM"])
w2 = stats.wilcoxon(b["LS NREM"], b["CA1 NREM"])
print(f"{len(b)} recordings with both: NREM rate LS median {b['LS NREM'].median():.3f}, CA1 median {b['CA1 NREM'].median():.3f};"
      f" LS < CA1 in {(b['LS NREM'] < b['CA1 NREM']).sum()}; Wilcoxon p = {w2.pvalue:.3g}")
print(f"WAKE rates: LS median {b['LS WAKE'].median():.3f}, CA1 median {b['CA1 WAKE'].median():.3f}")
print("by animal (median rates):")
print(rt.groupby("subject")[["LS NREM", "LS WAKE", "CA1 NREM", "CA1 WAKE"]].median().round(3).to_string())
fig, ax = plt.subplots(figsize=(4, 3))
for _, x in rt.iterrows():
    ax.plot([0, 1], [x["LS WAKE"], x["LS NREM"]], "m-", lw=0.5)
for _, x in b.iterrows():
    ax.plot([2, 3], [x["CA1 WAKE"], x["CA1 NREM"]], "k-", lw=0.5)
ax.set_xticks([0, 1, 2, 3], ["LS wake", "LS NREM", "CA1 wake", "CA1 NREM"]); ax.set_ylabel("events/s"); ax.set_ylim(0, 2)
plt.tight_layout(); plt.show()
print("rates above 2/s (outside the axis):", int((rt[["LS NREM", "LS WAKE"]] > 2).sum().sum() + (b[["CA1 NREM", "CA1 WAKE"]] > 2).sum().sum()))


**Result: partially reproduced.** LS HFO rates in NREM are far below CA1 ripple rates (medians 0.006
and 0.461 per s, lower in 34 of 39 recordings, Wilcoxon p = 3.1e-06). The detected LS events are
not more frequent in NREM than in WAKE: of 49 recordings with both states, NREM is higher in 18
and lower in 29 (Wilcoxon p = 0.44).

## fig5d-lag and fig5e-state-coupling

> LS HFOs are reliably coupled to CA1 ripples, with a median peak-to-peak lag of 6.4 +- 4.2 ms, CA1
> leading LS.

> HPC-LS coupling is stronger in NREM sleep than in waking; the fraction of LS HFOs within +-25 ms
> of a CA1 ripple is 0.3 in NREM and 0.15 in WAKE (P < 6.6e-6).

In [ ]:
# fig5d-lag and fig5e-state-coupling
def nearest(t, ref):
    ref = np.sort(ref)
    if len(ref) == 0:
        return np.full(len(t), np.nan)
    k = np.clip(np.searchsorted(ref, t), 1, len(ref) - 1) if len(ref) > 1 else np.zeros(len(t), int)
    lo, hi = ref[np.maximum(k - 1, 0)], ref[k]
    return np.where(np.abs(t - lo) <= np.abs(t - hi), t - lo, t - hi)   # t minus the nearest ref

lag_rows, all_lags, ccg_sum = [], [], 0
for r in R:
    if "hpc_events" not in r:
        continue
    d = nearest(r["ls_events"]["peak"], r["hpc_events"]["peak"])
    d = d[np.abs(d) <= 0.05] * 1000
    all_lags.append(d)
    if len(d):
        lag_rows.append({"file": r["path"].split("/")[1][:30], "subject": r["meta"]["subject"], "pairs": len(d), "median lag": np.median(d)})
    ccg_sum = ccg_sum + ls_hfo.ccg(np.sort(r["hpc_events"]["peak"]), np.sort(r["ls_events"]["peak"]), half=0.2, bin_s=0.005)
lags = pd.DataFrame(lag_rows)
pooled = np.concatenate(all_lags)
print(f"LS HFO minus nearest CA1 ripple within 50 ms: (a) median of {len(lags)} recording medians {lags['median lag'].median():.2f} ms"
      f" (SD {lags['median lag'].std():.2f}); (b) pooled median {np.median(pooled):.2f} ms over {len(pooled)} pairs  [paper 6.4 +- 4.2, CA1 leading]")
print(f"fraction of pairs with LS after CA1: {np.mean(pooled > 0):.3f}; at the same sample (|lag| < 0.5 ms): {np.mean(np.abs(pooled) < 0.5):.3f}")
nz = np.concatenate([d[np.abs(d) >= 1] for d in all_lags])
print(f"variant without pairs within 1 ms: pooled median {np.median(nz):.2f} ms over {len(nz)} pairs; LS after CA1 in {np.mean(nz > 0):.3f}")
print("fraction of pairs at the same sample, by animal:", {s_: round(float(np.mean(np.abs(np.concatenate([d for d, r in zip(all_lags, [r for r in R if 'hpc_events' in r]) if r['meta']['subject'] == s_])) < 0.5)), 3) for s_ in sorted({r['meta']['subject'] for r in R if 'hpc_events' in r})})
lagc = np.arange(-200, 201, 5)
flank = ccg_sum[np.abs(lagc) >= 100]
centre = ccg_sum[np.abs(lagc) <= 25]
print(f"pooled CA1-LS CCG (5 ms bins): peak within +-25 ms {centre.max():.0f} at {lagc[np.abs(lagc) <= 25][centre.argmax()]} ms;"
      f" flanks (100-200 ms) mean {flank.mean():.1f}, SD {flank.std():.1f}; peak - flank = {(centre.max() - flank.mean()) / flank.std():.1f} SD")
print("by animal, median of recording medians (ms):", lags.groupby("subject")["median lag"].median().round(2).to_dict())
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(lagc, ccg_sum, width=5, color="k"); axes[0].set_xlabel("LS HFO time - CA1 ripple time (ms)"); axes[0].set_ylabel("count")
axes[1].hist(pooled, bins=50, range=(-50, 50), color="m"); axes[1].set_xlabel("lag to nearest ripple (ms)")
plt.tight_layout(); plt.show()

rows = []
for r in R:
    st = r["meta"]["states"]
    if st is None or "hpc_events" not in r:
        continue
    e = r["ls_events"]
    d = np.abs(nearest(e["peak"], r["hpc_events"]["peak"]))
    row = {"file": r["path"].split("/")[1][:30], "subject": r["meta"]["subject"]}
    for lab, name in (("Non-REM", "NREM"), ("Awake", "WAKE")):
        sel = e["state"] == lab
        row[name + " n"] = int(sel.sum())
        row[name] = np.mean(d[sel] <= 0.025) if sel.sum() else np.nan
        row[name + " coupled"] = int((d[sel] <= 0.025).sum())
    rows.append(row)
cp = pd.DataFrame(rows)
cpk = cp[(cp["NREM n"] >= 10) & (cp["WAKE n"] >= 10)]
t = stats.ttest_ind(cpk.NREM, cpk.WAKE)
print(f"\n{len(cpk)} of {len(cp)} recordings with at least 10 LS HFOs in each state")
print(f"(a) mean over recordings: NREM {cpk.NREM.mean():.3f}, WAKE {cpk.WAKE.mean():.3f}; two-sample t-test p = {t.pvalue:.3g}; paired t-test p = {stats.ttest_rel(cpk.NREM, cpk.WAKE).pvalue:.3g} (variant)")
print(f"(b) pooled: NREM {cpk['NREM coupled'].sum() / cpk['NREM n'].sum():.3f}, WAKE {cpk['WAKE coupled'].sum() / cpk['WAKE n'].sum():.3f}  [paper 0.3 and 0.15]")
# chance: the same fraction with the CA1 ripple times shifted by 1 s (keeps the ripple rate)
ch = []
for r in R:
    st = r["meta"]["states"]
    if st is None or "hpc_events" not in r:
        continue
    e = r["ls_events"]
    d = np.abs(nearest(e["peak"], r["hpc_events"]["peak"] + 1.0))
    for lab in ("Non-REM", "Awake"):
        sel = e["state"] == lab
        ch.append({"state": lab, "frac": np.mean(d[sel] <= 0.025) if sel.sum() >= 10 else np.nan})
print("reference, ripple times shifted by 1 s (mean over recordings):", pd.DataFrame(ch).groupby("state").frac.mean().round(3).to_dict())
print(cpk.groupby("subject")[["NREM", "WAKE"]].mean().round(3).to_string())


**Result: fig5d-lag partially reproduced.** LS HFOs are coupled to CA1 ripples: the pooled
correlogram peaks at 0 ms, 166 SD above its flanks. The lag is 0.00 ms under both readings, outside
6.4 +- 4.2 ms, because 35% of the coupled pairs fall on the same sample (71% in DT8, 7% in DT9).
Without those pairs the pooled median is 3.2 ms, LS after CA1 in 55%; DT9, the animal with the
fewest same-sample pairs, gives 3.8 ms.

**Result: fig5e-state-coupling not reproduced.** Over 25 recordings, the fraction of LS HFOs within
25 ms of a CA1 ripple is 0.199 in NREM and 0.116 in WAKE (two-sample t-test p = 0.089); pooled,
0.088 and 0.041. The direction matches the paper, but the difference is not significant and the
values are below 0.3 and 0.15. Chance (ripple times shifted by 1 s) is 0.019 and 0.005.

## fig5f-ripple-triggered-firing and fig5f-no-phase-coherence

> LS firing around CA1 ripples is higher during NREM than during waking.

> CA1-LS phase coherence in the 120-180 Hz band around CA1 ripples does not differ from that of
> randomly jittered windows, so LS does not inherit the ripple wave by wave.

In [ ]:
# fig5f-ripple-triggered-firing and fig5f-no-phase-coherence
rows = []
for r in R:
    st = r["meta"]["states"]
    if st is None or "hpc_events" not in r:
        continue
    e = r["hpc_events"]
    tn, tw = e["peak"][e["state"] == "Non-REM"], e["peak"][e["state"] == "Awake"]
    if len(tn) < 20 or len(tw) < 20:
        continue
    for i, s in ls_hfo.sorted_spikes(r, "ls").items():
        rn = ls_hfo.counts_in(s, tn - 0.025, tn + 0.025).mean() / 0.05
        rw = ls_hfo.counts_in(s, tw - 0.025, tw + 0.025).mean() / 0.05
        base_n = rate_in(s, st, "Non-REM")
        base_w = rate_in(s, st, "Awake")
        rows.append({"subject": r["meta"]["subject"], "NREM": rn, "WAKE": rw, "NREM gain": rn / base_n if base_n > 0 else np.nan,
                     "WAKE gain": rw / base_w if base_w > 0 else np.nan})
rf = pd.DataFrame(rows)
t = stats.ttest_rel(rf.NREM, rf.WAKE)
print(f"{len(rf)} LS neurons (paper 1,313). Rate within +-25 ms of CA1 ripples: NREM mean {rf.NREM.mean():.2f} Hz, WAKE mean {rf.WAKE.mean():.2f} Hz;"
      f" NREM higher for {(rf.NREM > rf.WAKE).sum()} neurons; paired t-test p = {t.pvalue:.3g}")
g = rf.dropna(subset=["NREM gain", "WAKE gain"])
t2 = stats.ttest_rel(g["NREM gain"], g["WAKE gain"])
print(f"variant, rate divided by the neuron's mean rate in that state ({len(g)} neurons): NREM {g['NREM gain'].mean():.2f}, WAKE {g['WAKE gain'].mean():.2f}, paired t-test p = {t2.pvalue:.3g}")
print(rf.groupby("subject")[["NREM", "WAKE"]].agg(["count", "mean"]).round(2).to_string())

coh = [r["coherence"] for r in R if "coherence" in r and r["coherence"]["n"] > 0]
plv = np.array([c["plv"] for c in coh]); jit = np.array([c["jitter"] for c in coh])  # recordings x lags, recordings x 20 x lags
lagsc = coh[0]["lags"] * 1000
mid = len(lagsc) // 2
avg, javg = plv.mean(axis=0), jit.mean(axis=0)   # lags; 20 x lags
mu, sd = javg.mean(axis=0), javg.std(axis=0)
print(f"\n{len(coh)} recordings. CA1-LS 120-180 Hz phase-locking value at the ripple time: {avg[mid]:.4f};"
      f" jittered {mu[mid]:.4f} +- {sd[mid]:.4f} (SD of 20 sets); excess {(avg[mid] - mu[mid]) / sd[mid]:.1f} SD; bound mean + 3 SD = {mu[mid] + 3 * sd[mid]:.4f}")
print(f"lags within +-50 ms above the bound: {int((avg[np.abs(lagsc) <= 50] > (mu + 3 * sd)[np.abs(lagsc) <= 50]).sum())} of {int((np.abs(lagsc) <= 50).sum())}")
per = plv[:, mid] - jit[:, :, mid].mean(axis=1)
print(f"per recording, PLV minus mean jittered PLV: median {np.median(per):.4f}, above 0 in {(per > 0).sum()} of {len(per)}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(lagsc, avg, "m", label="CA1 ripples")
ax.fill_between(lagsc, mu - 3 * sd, mu + 3 * sd, color="k", alpha=0.3, label="jittered +-3 SD")
ax.set_xlabel("ms from CA1 ripple"); ax.set_ylabel("phase-locking value"); ax.legend(); plt.tight_layout(); plt.show()
print(f"plotted range: {min(avg.min(), (mu - 3 * sd).min()):.4f} to {max(avg.max(), (mu + 3 * sd).max()):.4f}")


**Result: fig5f-ripple-triggered-firing not reproduced.** Over 978 LS neurons, the rate within
25 ms of CA1 ripples is 1.39 Hz in NREM and 1.33 Hz in WAKE (paired t-test p = 0.24). Divided by
each neuron's rate in the same state, WAKE is higher (1.56 against 1.34, p = 0.00016).

**Result: fig5f-no-phase-coherence not reproduced.** The CA1-LS phase-locking value at the ripple
time is 0.405, against 0.202 +- 0.007 for jittered windows (31 SD above), and 90 of 125 lags within
+-50 ms exceed the bound. It is higher at ripples in 70 of 71 recordings. The same-sample events of
fig5d-lag suggest that part of the 120-200 Hz signal is common to the LS and hippocampal channels,
which would raise this coherence; the outputs do not separate that from a neural coupling.

## fig7e-ca1-spikes, fig7e-state and fig7e-duration

> The number of CA1 spikes in a SPW-R significantly predicts the number of spikes of 38% of LS
> neurons.

> Brain state (WAKE or NREM) in which a SPW-R occurs significantly predicts the spike count of 26%
> of LS neurons.

> The duration of hippocampal SPW-Rs significantly predicts the spike count of about 20% of LS
> neurons.

The model is fixed in the claims file (linear regression, 5-fold cross-validated MSE, 100 shuffles).
Reading (a) calls a neuron significant when its MSE is below the 5th percentile of the shuffles,
reading (b) when a t-test of the shuffles against it gives p < 0.05. The chance rate tests one
shuffle against the others.

In [ ]:
# fig7e-ca1-spikes, fig7e-state, fig7e-duration: does a ripple feature predict an LS neuron's spike count?
N_SHUF = 100


def fig7_rows(rng, fixed_window=None):
    rows = []
    for r in R:
        if "hpc_events" not in r:
            continue
        ca1 = ls_hfo.sorted_spikes(r, "hpc")
        lsu = ls_hfo.sorted_spikes(r, "ls")
        e = r["hpc_events"]
        if not ca1 or not lsu or len(e["peak"]) < 100:
            continue
        win = (e["start"], e["stop"]) if fixed_window is None else (e["peak"] - fixed_window, e["peak"] + fixed_window)
        Y = np.array([ls_hfo.counts_in(s, *win) for s in lsu.values()]).T.astype(float)   # ripples x neurons
        keep_n = Y.sum(axis=0) >= 20
        if not keep_n.any():
            continue
        Y = Y[:, keep_n]
        n_ca1 = np.sum([ls_hfo.counts_in(s, e["start"], e["stop"]) for s in ca1.values()], axis=0).astype(float)
        feats = {"ca1_spikes": (n_ca1, np.ones(len(n_ca1), bool)), "duration": (e["duration"], np.ones(len(n_ca1), bool)),
                 "state": ((e["state"] == "Non-REM").astype(float), np.isin(e["state"], ["Non-REM", "Awake"]))}
        for name, (x, sel) in feats.items():
            if sel.sum() < 100 or np.unique(x[sel]).size < 2:
                continue
            xs, ys = x[sel], Y[sel]
            folds = rng.permutation(np.arange(len(xs)) % 5)
            X = np.column_stack([xs] + [rng.permutation(xs) for _ in range(N_SHUF)])
            mse = ls_hfo.cv_mse(ys, X, folds)               # neurons x (1 + shuffles)
            actual, shuf = mse[:, 0], mse[:, 1:]
            for k in range(len(actual)):
                s = shuf[k]
                # chance: shuffle 0 tested against the other shuffles
                c, cs = s[0], s[1:]
                tt = stats.ttest_1samp(s, actual[k])
                tc = stats.ttest_1samp(cs, c)
                rows.append({"feature": name, "subject": r["meta"]["subject"], "file": r["path"].split("/")[1][:30],
                             "z": (s.mean() - actual[k]) / s.std(ddof=1) if s.std() > 0 else np.nan,
                             "sig_a": actual[k] < np.percentile(s, 5), "sig_b": (tt.pvalue < 0.05) and actual[k] < s.mean(),
                             "chance_a": c < np.percentile(cs, 5), "chance_b": (tc.pvalue < 0.05) and c < cs.mean()})
    return pd.DataFrame(rows)


def fig7_summary(f7):
    summary = []
    for name, d in f7.groupby("feature"):
        row = {"feature": name, "LS neurons": len(d), "recordings": d.file.nunique()}
        for rd in ("a", "b"):
            k, n, ch = d[f"sig_{rd}"].sum(), len(d), d[f"chance_{rd}"].mean()
            row[f"% ({rd})"] = round(100 * k / n, 1)
            row[f"chance % ({rd})"] = round(100 * ch, 1)
            row[f"binomial p vs chance ({rd})"] = stats.binomtest(int(k), n, max(ch, 1e-6), alternative="greater").pvalue
        summary.append(row)
    return pd.DataFrame(summary)


f7 = fig7_rows(np.random.default_rng(118))
f7s = fig7_summary(f7)
print(f7s.to_string(index=False))
print("paper: CA1 spikes 38%, state 26%, duration about 20%")
print("by animal, % significant under reading (a):")
print(f7.groupby(["feature", "subject"]).sig_a.agg(["count", "mean"]).assign(mean=lambda d: (100 * d["mean"]).round(1)).to_string())
fig, ax = plt.subplots(figsize=(5, 3))
for j, (name, d) in enumerate(f7.groupby("feature")):
    ax.plot(j + np.random.default_rng(1181).uniform(-0.2, 0.2, len(d)), d.z.clip(-5, 20), "k.", ms=2)
    ax.set_xticks(range(f7.feature.nunique()), sorted(f7.feature.unique()))
ax.set_ylabel("SDs of shuffled MSE above actual (clipped -5 to 20)"); plt.tight_layout(); plt.show()
print("z outside the axis (clipped):", int(((f7.z < -5) | (f7.z > 20)).sum()), "of", len(f7))

# Variant: spikes counted in a fixed +-25 ms window around the ripple time, so the count does not grow with duration.
f7v = fig7_rows(np.random.default_rng(1182), fixed_window=0.025)
print("variant, fixed +-25 ms window:")
print(fig7_summary(f7v).to_string(index=False))


**Result: fig7e-ca1-spikes partially reproduced.** The number of CA1 spikes predicts 79.4% of 880
LS neurons under reading (a) (chance 4.7%) and 88.4% under (b) (chance 40.5%), above chance and far
above the paper's 38%. With a fixed +-25 ms window, 24.7% under (a).

**Result: fig7e-state partially reproduced.** State predicts 34.3% of 813 LS neurons under (a)
(chance 5.4%) and 67.4% under (b) (chance 41.7%), above chance and above 26 +- 2.6%. With a fixed
window, 34.9% under (a).

**Result: fig7e-duration partially reproduced.** Duration predicts 86.0% of 880 LS neurons under
(a) (chance 5.9%) and 93.2% under (b), far above about 20%. Counting spikes from ripple start to
stop makes the count grow with duration; with a fixed +-25 ms window it is 17.8% under (a), within
tolerance, but the fixed window is a variant and does not decide the outcome.

## Summary

| Claim | Outcome |
|---|---|
| dataset-composition | partially reproduced |
| fig1c-hfo-each-animal | not reproduced |
| fig1c-event-count | reproduced |
| figs1-participation | not reproduced |
| fig1d-phase-locking | partially reproduced |
| fig1f-shorter-than-ripples | reproduced |
| fig1g-slower-than-ripples | not reproduced |
| fig1h-no-amplitude-frequency-correlation | not reproduced |
| fig2-confined-to-ls | not reproduced |
| fig2ef-intermediate-ls | blocked: metadata insufficient |
| fig3-travelling-waves | blocked: metadata insufficient |
| fig4b-pre-post-unchanged | reproduced |
| fig4b-hfo-above-behav | reproduced |
| fig4-sessions-few-change | blocked: other (7 recordings with 5 or more pairs) |
| fig4c-temporal-bias | blocked: other (4 recordings with 10 or more pairs) |
| fig4d-compression | not reproduced |
| figs2-post-rate-increase | partially reproduced |
| fig5b-nrem-and-rates | partially reproduced |
| fig5d-lag | partially reproduced |
| fig5e-state-coupling | not reproduced |
| fig5f-ripple-triggered-firing | not reproduced |
| fig5f-no-phase-coherence | not reproduced |
| fig6-optogenetic-hfos | blocked: data missing |
| fig7e-ca1-spikes | partially reproduced |
| fig7e-state | partially reproduced |
| fig7e-duration | partially reproduced |
| fig7e-ca1-depth | blocked: data missing |
| fig7e-replay | blocked: methods underspecified |
| fig7e-seqnmf | blocked: methods underspecified |
| fig7e-content-independent | blocked: methods underspecified |
| fig8-model | blocked: other |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's:

- The files hold no detected events, and the paper does not say which channels it used. LS HFOs and
  CA1 ripples are detected here on one channel each, chosen by a fixed rule, with the Methods'
  parameters and the buzcode detector's open details. The paper's noise channel is replaced by the
  EMG estimated from the LFP. Many detected LS events are not followed by LS firing, and a third
  of those coupled to ripples fall on the same sample as the ripple, so the event set differs from
  the paper's; most of the outcomes on HFOs follow from that.
- `hpc` units and channels are taken as CA1. CA3 recordings are left out of ripple analyses, as
  their CA1 reference channel is not in the files.
- Where the paper does not state a test or model (Figure 1H, Figure 7E, the pair inclusion of
  Figure 4), the choice was fixed in the claims file before the analysis.

What was found about the dandiset:

- No detected HFOs or ripples, no detection channels, and no unit waveforms or channels are
  deposited. Electrode coordinates are NaN in every file.
- Nine files have units without a region label (268 units); the `hpc` label does not separate CA1.
- 25 units have spike times out of order.
- The CA3 recordings have no CA1 reference channel; the optogenetic rats are not in the dandiset.
- Spikes, LFP, sleep states and trials are consistent with one clock.